# 06 — Export assets

The maps and the animation already exist. This is the part that frames them for
somewhere specific — a reel, a post, a figure in an essay — with the
visualization on its own and none of the site's interface around it.

Nothing here writes into the repository. Everything lands on the Desktop.

In [ ]:
%load_ext autoreload
%autoreload 2

from schematic import feeds, loom, pipeline, animate
from schematic.linegraph import LineGraph
from schematic.crs import to_mercator
from schematic.render import render, octilinearity, Style

FEED = "la-metro-rail"
LINE_ORDER = list("ABCDEK")   # the order lines are drawn in, back to front

In [ ]:
from schematic import export

for p in export.PRESETS.values():
    size = f"{p.width}x{p.height}" if p.width else "native"
    print(f"{p.name:18} {p.platform:11} {size:>11}  {p.kind}")

### Presentation mode is a URL, not a mode of this notebook

The animation page reads its own parameters, so the exporter is only another
caller. The same URL opened by hand gives you a clean full-screen view to
present from or to screen-record yourself.

In [ ]:
print(export.url_for(FEED, export.PRESETS["instagram-reel"], at="07:30"))

### The frame is padded inside the SVG

A preset's aspect ratio is reached by growing the `viewBox`, never by
letterboxing the SVG inside the page — the page background and the map
background differ in the dark theme, so a letterbox seams.

Growing, never cropping: cropping a transit map cuts off real stations. The
split is deliberately uneven, because on a tall frame the added height is the
gutter the title and clock live in.

In [ ]:
box = (-31, -32.58, 1717.82, 1223.63)          # LA's own viewBox
for name in ["instagram-reel", "instagram-post", "linkedin"]:
    p = export.PRESETS[name]
    x, y, w, h = export.padded_box(box, p.aspect, p.frame_top)
    print(f"{name:16} {w/h:.3f}  above {-y - box[1]:7.0f}  below {(y+h)-(box[1]+box[3]):7.0f}")

### Storyboards

A video is a list of beats. Anything a beat does not name carries over from the
one before, so the sequences stay short.

A beat that `sweep`s owns the clock outright and runs it across a window. That
matters: a whole service day stepped across ten seconds moves the clock about
150 simulated seconds per frame, and trains teleport rather than move. The
storyboards sweep the morning peak instead, and a test fails any sweep above 60
seconds per frame.

In [ ]:
for name, beats in export.STORYBOARDS.items():
    total = sum(b.secs for b in beats)
    rate = max([export.sweep_rate(b, 30, (0, 86400)) for b in beats if b.sweep] or [0])
    print(f"{name:8} {total:5.1f}s  {export.frame_count(beats, 30):4d} frames"
          + (f"  sweep {rate:.0f}s/frame" if rate else ""))

### Make something

A still first, because it is quick. The capture supersamples and resamples down
to the preset's exact size: a platform expecting 1080 wide does better with a
clean 1080 than with a 2160 it downscales itself, and these maps are mostly
one-pixel strokes.

In [ ]:
paths = export.run(FEED, "instagram-post", at="08:00", quality="draft")
for p in paths:
    print(p, f"{p.stat().st_size/1024:.0f} KB")

### Every file explains itself

A picture travels further than the page it came from, so each export carries the
network's own caveats and a piece of alt text alongside it. The project argues
that information people cannot read is not accessible; shipping an image with no
description would undercut that.

In [ ]:
import json
print(json.dumps(json.loads(paths[0].with_suffix(paths[0].suffix + ".json").read_text()),
                 indent=2))

### And a video

The page's clock is stepped by hand, one `advance(1/fps)` per captured frame,
rather than recorded in real time. That is what makes a run reproducible — the
same command twice gives byte-identical output — and it is also why a
transition can land exactly on a frame boundary instead of being cut off
mid-morph.

Reckon on 40–70 ms a frame, so about a minute for a 25-second clip.

In [ ]:
video = export.run(FEED, "portfolio-gif", quality="draft")[0]
print(video, f"{video.stat().st_size/1024:.0f} KB")

In [ ]:
from IPython.display import Image
Image(str(video))

### From the shell

Everything above is `bin/export`, which is the way you will actually use it.

```bash
bin/export --list
bin/export cdmx-metro -p instagram-reel --storyboard reveal
bin/export nyc-subway -p portfolio-svg
bin/export la-metro-rail -p all --sheet --open
```